# samebit on an NVIDIA GPU

Settings → Accelerator → **GPU T4 x2** (or P100), Internet **on**. Then *Run all*. The last cell prints a summary to copy back.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
!git clone -q https://github.com/Shakhtar-Sankur/samebit
%cd samebit
!pip install -q -e ".[test]"
try:
    import cupy
except ImportError:
    !pip install -q cupy-cuda12x
import cupy

print("cupy", cupy.__version__)

In [ ]:
# Conformance: every installed backend, including CUDA, against the golden digest
!python -m samebit

In [ ]:
# Split-K: each dot product is split across 1 to 256 thread blocks and combined
# with atomicAdd, in a different order on every run
import os
import subprocess

for k in (1, 2, 8, 32, 256):
    out = subprocess.run(
        ["python", "-m", "samebit", "cuda"],
        env={**os.environ, "SAMEBIT_CUDA_SPLITK": str(k)},
        capture_output=True,
        text=True,
    ).stdout
    print(f"splitk={k:<4}", out.strip().splitlines()[-1])

In [ ]:
# Op-level and end-to-end tests on every backend
!python -m pytest -q

In [ ]:
import subprocess

print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True
    ).stdout
)
print(subprocess.run(["python", "-m", "samebit"], capture_output=True, text=True).stdout)